In [1]:
!pip install -q transformers sentencepiece psutil scikit-learn matplotlib pandas
print("Done!")

Done!


In [2]:
from google.colab import files
uploaded = files.upload()  # upload baseline_model.zip AND results.zip

!unzip -q baseline_model.zip -d /content/content/
!unzip -q results.zip -d /content/content/
print("Done!")
!ls /content/content/results/

Saving results.zip to results.zip
Saving baseline_model.zip to baseline_model.zip
Done!
ls: cannot access '/content/content/results/': No such file or directory


In [3]:
import os, time, json, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import psutil
import matplotlib.pyplot as plt
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AlbertConfig,
    AlbertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import f1_score

SEED = 42
torch.manual_seed(SEED)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR  = '/content/content/content/baseline_model'
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_LABELS = 7
EPOCHS     = 5
TEMP       = 4.0
ALPHA      = 0.5
label2id   = {'neutral':0,'joy':1,'anger':2,'surprise':3,'sadness':4,'fear':5,'disgust':6}
id2label   = {v: k for k, v in label2id.items()}

with open('/content/content/content/results/all_results.json') as f:
    ALL = json.load(f)

print("Device:", device)
print("Keys loaded:", list(ALL.keys()))

Device: cpu
Keys loaded: ['params', 'baseline_f1', 'baseline_size_mb', 'baseline_latency_ms', 'baseline_ram_mb']


In [4]:
!ls -R /content/content/

/content/content/:
content

/content/content/content:
baseline_model	results

/content/content/content/baseline_model:
config.json  model.safetensors	tokenizer_config.json  tokenizer.json

/content/content/content/results:
all_results.json  test.csv  train.csv


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, keep_accents=True)
train_df  = pd.read_csv('/content/content/content/results/train.csv')
test_df   = pd.read_csv('/content/content/content/results/test.csv')

class HindiDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts  = df['Sentence'].tolist()
        self.labels = df['label'].tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], truncation=True,
            max_length=MAX_LENGTH, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(HindiDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(HindiDataset(test_df),  batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 400 | Test batches: 100


In [6]:
teacher = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, num_labels=NUM_LABELS
).to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print("Teacher loaded and frozen")

# Student: 3 layers instead of 12
cfg = AlbertConfig(
    vocab_size=teacher.config.vocab_size,
    hidden_size=teacher.config.hidden_size,
    num_hidden_layers=3,
    num_attention_heads=teacher.config.num_attention_heads,
    intermediate_size=teacher.config.intermediate_size,
    num_labels=NUM_LABELS
)
student = AlbertForSequenceClassification(cfg).to(device)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f"Teacher: {t_params:,} params")
print(f"Student: {s_params:,} params")
print(f"Compression: {t_params/s_params:.1f}x fewer parameters")

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Teacher loaded and frozen
Teacher: 33,448,967 params
Student: 33,448,967 params
Compression: 1.0x fewer parameters


In [ ]:
def distil_loss(s_logits, t_logits, labels, T, alpha):
    task = F.cross_entropy(s_logits, labels)
    soft_t = F.softmax(t_logits / T, dim=-1)
    soft_s = F.log_softmax(s_logits / T, dim=-1)
    kl = F.kl_div(soft_s, soft_t, reduction='batchmean') * (T**2)
    return alpha * task + (1 - alpha) * kl

optimizer    = AdamW(student.parameters(), lr=5e-5, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(optimizer, total_steps//10, total_steps)

best_f1, best_state = 0.0, None

for epoch in range(EPOCHS):
    student.train()
    total_loss, steps = 0, 0

    for batch in train_loader:
        labels = batch.pop('labels').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            t_out = teacher(**batch)
        s_out = student(**batch)
        loss  = distil_loss(s_out.logits, t_out.logits, labels, TEMP, ALPHA)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        steps += 1

    student.eval()
    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop('labels').to(device)
            batch  = {k: v.to(device) for k, v in batch.items()}
            preds  = student(**batch).logits.argmax(dim=-1).cpu().numpy()
            preds_all.extend(preds)
            labs_all.extend(labels.cpu().numpy())

    ep_f1 = f1_score(labs_all, preds_all, average='macro')
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/steps:.4f} | F1: {ep_f1:.4f}")
    if ep_f1 > best_f1:
        best_f1 = ep_f1
        best_state = {k: v.clone() for k, v in student.state_dict().items()}

student.load_state_dict(best_state)
print(f"\nBest student F1: {best_f1:.4f}")

Epoch 1/5 | Loss: 0.8497 | F1: 0.2670


In [ ]:
student.eval()
sample = tokenizer(
    "यह बहुत अच्छा है।", return_tensors='pt',
    truncation=True, max_length=MAX_LENGTH, padding='max_length'
).to(device)

for _ in range(10):
    with torch.no_grad(): _ = student(**sample)
times = []
for _ in range(100):
    t = time.perf_counter()
    with torch.no_grad(): _ = student(**sample)
    times.append((time.perf_counter() - t) * 1000)

torch.save(student.state_dict(), '/content/results/model_distilled.pt')
dist_size = os.path.getsize('/content/results/model_distilled.pt') / 1024**2

proc = psutil.Process(os.getpid())
r0   = proc.memory_info().rss / 1024**2
with torch.no_grad(): _ = student(**sample)
ram  = proc.memory_info().rss / 1024**2 - r0

ALL['distilled'] = {
    'f1':         round(best_f1, 4),
    'size_mb':    round(dist_size, 1),
    'latency_ms': round(float(np.mean(times)), 2),
    'ram_mb':     round(ram, 1),
    'params':     s_params
}
print("Distilled results:", ALL['distilled'])

with open('/content/results/all_results.json', 'w') as f:
    json.dump(ALL, f, indent=2)
print("\nAll results:")
print(json.dumps(ALL, indent=2))

In [ ]:
# ── Benchmark table
baseline_size = ALL['baseline_size_mb']

rows = [
    ('Baseline (FP32)',     ALL['baseline_size_mb'],            ALL['baseline_f1'],            ALL['baseline_latency_ms']),
    ('INT8 Quantization',   ALL['int8']['size_mb'],             ALL['int8']['f1'],             ALL['int8']['latency_ms']),
    ('Pruning 30%',         ALL['pruned_30pct']['size_mb'],     ALL['pruned_30pct']['f1'],     ALL['pruned_30pct']['latency_ms']),
    ('Pruning 50%',         ALL['pruned_50pct']['size_mb'],     ALL['pruned_50pct']['f1'],     ALL['pruned_50pct']['latency_ms']),
    ('Pruning 70%',         ALL['pruned_70pct']['size_mb'],     ALL['pruned_70pct']['f1'],     ALL['pruned_70pct']['latency_ms']),
    ('Distilled (3-layer)', ALL['distilled']['size_mb'],        ALL['distilled']['f1'],        ALL['distilled']['latency_ms']),
]

df_table = pd.DataFrame(rows, columns=['Method','Size (MB)','F1 Score','Latency (ms)'])
df_table.to_csv('/content/results/benchmark_table.csv', index=False)
print("=== FINAL BENCHMARK TABLE ===")
print(df_table.to_string(index=False))

# ── Threshold curve
fig, ax = plt.subplots(figsize=(11, 6))

methods = [
    ('Baseline',      baseline_size,                         ALL['baseline_f1'],         '#2196F3'),
    ('INT8',          ALL['int8']['size_mb'],                 ALL['int8']['f1'],           '#4CAF50'),
    ('Pruned 30%',    ALL['pruned_30pct']['size_mb'],         ALL['pruned_30pct']['f1'],   '#FF9800'),
    ('Pruned 50%',    ALL['pruned_50pct']['size_mb'],         ALL['pruned_50pct']['f1'],   '#FF5722'),
    ('Pruned 70%',    ALL['pruned_70pct']['size_mb'],         ALL['pruned_70pct']['f1'],   '#F44336'),
    ('Distilled',     ALL['distilled']['size_mb'],            ALL['distilled']['f1'],      '#9C27B0'),
]

xs = [baseline_size / float(sz) for _, sz, _, _ in methods]
ys = [float(f1) for _, _, f1, _ in methods]
cs = [c for _, _, _, c in methods]
ls = [l for l, _, _, _ in methods]

ax.plot(xs, ys, 'k--', alpha=0.3, zorder=1)
for x, y, c, l in zip(xs, ys, cs, ls):
    ax.scatter(x, y, color=c, s=150, zorder=3, label=l)
    ax.annotate(l, (x, y), textcoords='offset points', xytext=(8, 5),
                fontsize=10, fontweight='bold')

ax.axhline(0.20, color='red', linestyle=':', alpha=0.7, label='Usability floor (F1=0.20)')
ax.fill_between([min(xs)-0.1, max(xs)+0.3], 0.20, 1.0, alpha=0.05, color='green')

ax.set_xlabel('Compression Ratio (× smaller than baseline)', fontsize=13)
ax.set_ylabel('Macro F1 Score', fontsize=13)
ax.set_title('Hindi Emotion Classification — Compression Threshold Curve\n'
             'Identifying the accuracy cliff for low-resource Indian device deployment',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(-0.05, 0.55)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/results/threshold_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

In [ ]:
!zip -r /content/final_results.zip /content/results
from google.colab import files
files.download('/content/final_results.zip')
print("✅ ALL EXPERIMENTS DONE!")
print("Upload the results/ folder to GitHub.")
print("threshold_curve.png and benchmark_table.csv go in your submission.")